# E-Commerce Sales & Customer Analytics

**Objective:** Explore sales performance, product profitability, customer value and regional trends using Python.

Dataset: synthetic practice dataset generated for this portfolio project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw = pd.read_csv('../data/ecommerce_sales_raw.csv', parse_dates=['order_date'])
raw.head()

## 1. Data quality checks

In [ ]:
print('Shape:', raw.shape)
print('\nMissing values:\n', raw.isna().sum())
print('\nDuplicate order IDs:', raw['order_id'].duplicated().sum())
print('\nData types:\n', raw.dtypes)

## 2. Clean the data

In [ ]:
df = raw.drop_duplicates(subset=['order_id']).copy()
df['region'] = df['region'].fillna('Unknown')
df['discount_pct'] = df['discount_pct'].fillna(0)
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.to_period('M').astype(str)
df['profit_margin_pct'] = np.where(df['revenue'] != 0, df['profit'] / df['revenue'] * 100, 0)
df.to_csv('../data/ecommerce_sales_clean.csv', index=False)
df.head()

## 3. KPI summary

In [ ]:
kpis = {
    'Orders': df['order_id'].nunique(),
    'Customers': df['customer_id'].nunique(),
    'Revenue': round(df['revenue'].sum(), 2),
    'Profit': round(df['profit'].sum(), 2),
    'Average Order Value': round(df['revenue'].sum() / df['order_id'].nunique(), 2),
    'Profit Margin %': round(df['profit'].sum() / df['revenue'].sum() * 100, 2)
}
pd.Series(kpis, name='Value')

## 4. Sales trends

In [ ]:
monthly = df.groupby('month', as_index=False).agg(revenue=('revenue','sum'), profit=('profit','sum'))
monthly.plot(x='month', y='revenue', kind='line', marker='o', figsize=(12,5), title='Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Category and regional performance

In [ ]:
category = df.groupby('category', as_index=False).agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','nunique')).sort_values('revenue', ascending=False)
region = df.groupby('region', as_index=False).agg(revenue=('revenue','sum'), profit=('profit','sum')).sort_values('revenue', ascending=False)
display(category)
display(region)

## 6. Customer RFM segmentation

In [ ]:
snapshot = df['order_date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot-x.max()).days),
    frequency=('order_id','nunique'),
    monetary=('revenue','sum')
).reset_index()
rfm['R_score'] = pd.qcut(rfm['recency'].rank(method='first'), 4, labels=[4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['RFM_score'] = rfm[['R_score','F_score','M_score']].sum(axis=1)
rfm['segment'] = np.select([rfm.RFM_score>=10, rfm.RFM_score>=7], ['High Value','Potential Loyal'], default='Needs Attention')
rfm.sort_values('monetary', ascending=False).head(10)

## 7. Discount and channel analysis

In [ ]:
discount_analysis = df.groupby('discount_pct', as_index=False).agg(orders=('order_id','nunique'), revenue=('revenue','sum'), profit=('profit','sum'))
channel_analysis = df.groupby('sales_channel', as_index=False).agg(orders=('order_id','nunique'), revenue=('revenue','sum'), profit=('profit','sum'))
display(discount_analysis)
display(channel_analysis)

## 8. Export analysis tables

In [ ]:
category.to_csv('../reports/category_performance.csv', index=False)
region.to_csv('../reports/region_performance.csv', index=False)
rfm.to_csv('../reports/customer_rfm_segments.csv', index=False)
monthly.to_csv('../reports/monthly_sales_trend.csv', index=False)
print('Analysis tables exported to reports/.')

## 9. Business questions to answer

1. Which category contributes the most revenue and profit?
2. Which region has the strongest sales performance?
3. How does average order value change month to month?
4. Which customers are high-value based on RFM?
5. Are higher discounts associated with weaker profit margins?
6. Which sales channel contributes the most revenue?

**Important:** Treat findings as descriptive, not causal. The dataset is synthetic and intended for portfolio practice.